In [17]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import lightgbm as lgb
import joblib
import time

In [3]:
df = pd.read_csv("../../datasets/training_dataset/train.csv")

In [4]:
X = df.drop(columns=['label'])
y = df['label']

In [5]:
print("\nTraining Decision Tree...")
start = time.time()
dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X, y)
y_pred_dt = dt.predict(X)
end = time.time()
print(f"Decision Tree Training Time: {end - start:.2f} sec")


Training Decision Tree...
Decision Tree Training Time: 90.76 sec


In [15]:
print("\nDecision Tree Report:")
print("Accuracy:", accuracy_score(y, y_pred_dt))
print(classification_report(y, y_pred_dt))
joblib.dump(dt, "./models/decision_tree_model_with_no_feature_filtration.pkl")


Decision Tree Report:
Accuracy: 0.9026110416666666
              precision    recall  f1-score   support

           0       0.90      0.92      0.91   2696949
           1       0.90      0.88      0.89   2103051

    accuracy                           0.90   4800000
   macro avg       0.90      0.90      0.90   4800000
weighted avg       0.90      0.90      0.90   4800000



['./models/decision_tree_model_with_no_feature_filtration.pkl']

In [11]:
print("\nTraining Random Forest...")
start = time.time()
rf = RandomForestClassifier(
    n_estimators=100,   # number of trees (default 100)
    max_depth=None,     # let trees expand fully
    n_jobs=-1,          # use all CPU cores
    random_state=42
)
rf.fit(X, y)
y_pred_rf = rf.predict(X)
end = time.time()
print(f"Random Forest Training Time: {end - start:.2f} sec")


Training Random Forest...
Random Forest Training Time: 336.97 sec


In [ ]:
print("\nRandom Forest Report:")
print("Accuracy:", accuracy_score(y, y_pred_rf))
print(classification_report(y, y_pred_rf))
# joblib.dump(rf, "./models/random_forest_model_with_no_feature_filtration.pkl")


Random Forest Report:
Accuracy: 0.9955745833333334
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   2696949
           1       1.00      0.99      0.99   2103051

    accuracy                           1.00   4800000
   macro avg       1.00      1.00      1.00   4800000
weighted avg       1.00      1.00      1.00   4800000



['./models/random_forest_model_with_no_feature_filtration.pkl']

Random forest is rejected though it gives good accuracy but is too much memory intensive.

In [ ]:
train_data = lgb.Dataset(X, label=y)

params = {
    "objective": "binary",  
    "boosting": "gbdt",
    "metric": "binary_error", 
    "num_leaves": 64,
    "learning_rate": 0.1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1
}

print("\nTraining LightGBM on full train.csv ...")
start = time.time()

# Train model (no validation split)
model = lgb.train(
    params,
    train_data,
    num_boost_round=500
)

end = time.time()
print(f"✅ LightGBM Training Time: {end - start:.2f} sec")

# Predictions on training data
y_pred = model.predict(X)
y_pred_binary = (y_pred > 0.5).astype(int)


Training LightGBM on full train.csv ...
✅ LightGBM Training Time: 69.17 sec


In [20]:
print("\nLightGBM Report (Train Set):")
print("Accuracy:", accuracy_score(y, y_pred_binary))
print(classification_report(y, y_pred_binary))
model.save_model("./models/lightgbm_model.txt")



LightGBM Report (Train Set):
Accuracy: 0.94526125
              precision    recall  f1-score   support

           0       0.95      0.95      0.95   2696949
           1       0.94      0.94      0.94   2103051

    accuracy                           0.95   4800000
   macro avg       0.94      0.94      0.94   4800000
weighted avg       0.95      0.95      0.95   4800000

